In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')
# Add notebook cells here to replace solution/__init__.py and your kernels.
# The prepared starter is retained when you run this template unchanged.


In [ ]:
# Generated solution installer. Writes files only; does not import solution or run training.
# Payload SHA256: c7d4d4584ce22c6e597fb634a20bcd2a933e798ef528635d0ba9bb729016d1df
# Manifest SHA256: a4bb0fb70b5e544bb8b3f02993ea509e5a79d736ea1c63d8583a213558cfe216
def _soup_install_solution_c7d4d4584ce22c6e(payload_base64, expected_payload_sha256, expected_manifest_sha256):
    """Self-contained runtime; its source is embedded in the inserted code cell."""
    import base64
    import hashlib
    import json
    import os
    from pathlib import Path, PurePosixPath
    import stat
    import uuid

    def canonical(value):
        return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")

    def digest(data):
        return hashlib.sha256(data).hexdigest()

    raw_repo = os.environ.get("SOUP_REPO")
    if not raw_repo or not raw_repo.strip():
        raise RuntimeError("SOUP_REPO must be set by the service template before this cell runs")
    repo = Path(raw_repo)
    if not repo.is_absolute() or ".." in repo.parts:
        raise RuntimeError("SOUP_REPO must be an absolute repository path without '..'")

    payload_bytes = base64.b64decode(payload_base64, validate=True)
    if digest(payload_bytes) != expected_payload_sha256:
        raise RuntimeError("Embedded solution payload SHA256 mismatch")
    payload = json.loads(payload_bytes)
    if not isinstance(payload, dict) or set(payload) != {"schema_version", "files"}:
        raise ValueError("Invalid solution payload schema")
    if type(payload["schema_version"]) is not int or payload["schema_version"] != 1:
        raise ValueError("Unsupported solution payload version")
    if not isinstance(payload["files"], list) or not payload["files"]:
        raise ValueError("Solution payload must contain source files")

    decoded, manifest, names = [], [], set()
    for item in payload["files"]:
        if not isinstance(item, dict) or set(item) != {"path", "size_bytes", "sha256", "data_base64"}:
            raise ValueError("Invalid solution file record")
        name = item["path"]
        if not isinstance(name, str) or not name or "\\" in name or "\x00" in name:
            raise ValueError("Invalid solution path")
        path = PurePosixPath(name)
        if (path.is_absolute() or str(path) != name or len(path.parts) < 2
                or path.parts[0] != "solution" or any(p in (".", "..") for p in path.parts)
                or any(":" in p for p in path.parts) or path.suffix != ".py"):
            raise ValueError(f"Unsafe solution path: {name!r}")
        if name in names:
            raise ValueError(f"Duplicate solution path: {name}")
        names.add(name)
        data = base64.b64decode(item["data_base64"], validate=True)
        if type(item["size_bytes"]) is not int or item["size_bytes"] != len(data):
            raise ValueError(f"Source size mismatch: {name}")
        if item["sha256"] != digest(data):
            raise ValueError(f"Source SHA256 mismatch: {name}")
        manifest.append({key: item[key] for key in ("path", "size_bytes", "sha256")})
        decoded.append((path.parts[1:], data, name))
    for name in names:
        if any(str(parent) in names for parent in PurePosixPath(name).parents):
            raise ValueError(f"Conflicting file and directory paths: {name}")
    if digest(canonical(manifest)) != expected_manifest_sha256:
        raise RuntimeError("Embedded solution manifest SHA256 mismatch")

    # Directory descriptors plus O_NOFOLLOW prevent symlink traversal even
    # between checking a path and opening it. The service runtime is Linux.
    if not hasattr(os, "O_NOFOLLOW") or not hasattr(os, "O_DIRECTORY"):
        raise RuntimeError("Safe solution installation requires Linux directory descriptors")
    directory_flags = os.O_RDONLY | os.O_DIRECTORY | os.O_NOFOLLOW
    file_flags = os.O_RDONLY | os.O_NOFOLLOW

    def open_repo():
        descriptor = os.open(repo.anchor, directory_flags)
        try:
            for name in repo.parts[1:]:
                next_descriptor = os.open(name, directory_flags, dir_fd=descriptor)
                os.close(descriptor)
                descriptor = next_descriptor
            return descriptor
        except BaseException:
            os.close(descriptor)
            raise

    def open_parent(solution_descriptor, parts, create):
        descriptor = os.dup(solution_descriptor)
        try:
            for name in parts[:-1]:
                if create:
                    try:
                        os.mkdir(name, mode=0o755, dir_fd=descriptor)
                    except FileExistsError:
                        pass
                try:
                    next_descriptor = os.open(name, directory_flags, dir_fd=descriptor)
                except FileNotFoundError:
                    if not create:
                        os.close(descriptor)
                        return None
                    raise
                os.close(descriptor)
                descriptor = next_descriptor
            return descriptor
        except BaseException:
            os.close(descriptor)
            raise

    def check_leaf(parent, leaf, name):
        try:
            mode = os.stat(leaf, dir_fd=parent, follow_symlinks=False).st_mode
        except FileNotFoundError:
            return
        if not stat.S_ISREG(mode):
            raise RuntimeError(f"Destination conflict (symlink or non-file): {name}")

    def matches(parent, leaf, data):
        try:
            descriptor = os.open(leaf, file_flags, dir_fd=parent)
        except FileNotFoundError:
            return False
        with os.fdopen(descriptor, "rb") as stream:
            status = os.fstat(stream.fileno())
            if not stat.S_ISREG(status.st_mode):
                raise RuntimeError(f"Destination is not a regular file: {leaf}")
            if status.st_size != len(data):
                return False
            actual = hashlib.sha256()
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                actual.update(chunk)
            return actual.hexdigest() == digest(data)

    repo_descriptor = open_repo()
    solution_descriptor = None
    written = unchanged = 0
    try:
        try:
            os.mkdir("solution", mode=0o755, dir_fd=repo_descriptor)
        except FileExistsError:
            pass
        solution_descriptor = os.open("solution", directory_flags, dir_fd=repo_descriptor)
        # Check every existing destination before replacing any file.
        for parts, data, name in decoded:
            parent = open_parent(solution_descriptor, parts, create=False)
            if parent is not None:
                try:
                    check_leaf(parent, parts[-1], name)
                finally:
                    os.close(parent)
        for parts, data, name in decoded:
            parent = open_parent(solution_descriptor, parts, create=True)
            temporary = None
            try:
                check_leaf(parent, parts[-1], name)
                if matches(parent, parts[-1], data):
                    unchanged += 1
                    continue
                temporary = ".soup-install-" + uuid.uuid4().hex + ".tmp"
                descriptor = os.open(temporary, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW,
                                     0o644, dir_fd=parent)
                with os.fdopen(descriptor, "wb") as stream:
                    stream.write(data)
                    stream.flush()
                    os.fsync(stream.fileno())
                check_leaf(parent, parts[-1], name)
                os.replace(temporary, parts[-1], src_dir_fd=parent, dst_dir_fd=parent)
                temporary = None
                written += 1
            finally:
                if temporary is not None:
                    try:
                        os.unlink(temporary, dir_fd=parent)
                    except FileNotFoundError:
                        pass
                os.close(parent)
        # Verify every installed file against the hashed manifest, including
        # files that were already identical and therefore not rewritten.
        for parts, data, name in decoded:
            parent = open_parent(solution_descriptor, parts, create=False)
            if parent is None:
                raise RuntimeError(f"Installed source directory disappeared: {name}")
            try:
                check_leaf(parent, parts[-1], name)
                if not matches(parent, parts[-1], data):
                    raise RuntimeError(f"Installed source SHA256 mismatch: {name}")
            finally:
                os.close(parent)
    finally:
        if solution_descriptor is not None:
            os.close(solution_descriptor)
        os.close(repo_descriptor)
    result = {"payload_sha256": expected_payload_sha256,
              "manifest_sha256": expected_manifest_sha256,
              "files": len(decoded), "written": written, "unchanged": unchanged}
    print(json.dumps({"soup_solution_install": result}, sort_keys=True))
    return result

_soup_install_solution_c7d4d4584ce22c6e(
    '...',
)
del _soup_install_solution_c7d4d4584ce22c6e


In [ ]:
# Run only the solution. The server reuses its saved matching public baseline.
run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '900', '--only', 'solution'], cwd=repo, check=True)


In [ ]:
# Export small reports and logs. Large adapter/checkpoint files remain in this job's VM workspace.
for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())
# This is a public-data screen. Notebook output does not establish official admission or score.
